# 05 — Early ViT-3D baseline: threshold and calibration analysis

Analysis of the out-of-fold patient predictions of the first 3D baseline (`results/vit3d_patient_predictions.csv`, 170 patients). The predicted probabilities concentrate around 0.5 and the AUC is 0.555, i.e. this baseline is close to chance. It is kept as a documented negative result; the confusion matrix computed from these predictions is in `results/fig_confusion_matrix_vit3d.png`.

Paths are Windows paths from the original machine (`D:\\NSCLC_PROJECT\\...`); change the path constants at the top of each notebook before running. Raw data and model weights are **not** included in this repository.

In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score, balanced_accuracy_score, roc_curve

# ۱. خواندن پیش‌بینی‌های ذخیره‌شده
pred_df = pd.read_csv(r"D:\NSCLC_PROJECT\vit3d_patient_predictions.csv")

# ۲. پیدا کردن بهترین Threshold با استفاده از Youden's J Statistic روی کل پیش‌بینی‌ها
fpr, tpr, thresholds = roc_curve(pred_df["label"], pred_df["prob"])
youden_j = tpr - fpr
best_idx = np.argmax(youden_j)
best_threshold = thresholds[best_idx]

print(f"بهترین Threshold به دست آمده (Youden Index): {best_threshold:.4f}")

# ۳. اعمال Threshold جدید برای محاسبه متریک‌های اصلاح‌شده
preds_optimized = (pred_df["prob"] >= best_threshold).astype(int)

final_acc = accuracy_score(pred_df["label"], preds_optimized)
final_bal_acc = balanced_accuracy_score(pred_df["label"], preds_optimized)
final_auc = roc_auc_score(pred_df["label"], pred_df["prob"])
final_f1 = f1_score(pred_df["label"], preds_optimized, zero_division=0)

print("\n========== نتایج نهایی اصلاح‌شده با Threshold بهینه ==========")
print(f"Accuracy         : {final_acc:.3f}")
print(f"Balanced Accuracy: {final_bal_acc:.3f}")
print(f"AUC              : {final_auc:.3f}")
print(f"F1-Score         : {final_f1:.3f}")

# ۴. ذخیره نتایج جدید در صورت نیاز
pred_df["pred_optimized"] = preds_optimized
pred_df.to_csv(r"D:\NSCLC_PROJECT\vit3d_patient_predictions_optimized.csv", index=False)

بهترین Threshold به دست آمده (Youden Index): 0.4998

========== نتایج نهایی اصلاح‌شده با Threshold بهینه ==========
Accuracy         : 0.776
Balanced Accuracy: 0.581
AUC              : 0.555
F1-Score         : 0.296


In [2]:
from sklearn.calibration import CalibratedClassifierCV
from sklearn.linear_model import LogisticRegression
import numpy as np
import pandas as pd

# خواندن فایل پیش‌بینی‌های OOF موجود
pred_df = pd.read_csv(r"D:\NSCLC_PROJECT\vit3d_patient_predictions.csv")

# کالیبراسیون احتمال با رگرسیون لجستیک (Platt Scaling)
lr = LogisticRegression()
X = pred_df["prob"].values.reshape(-1, 1)
y = pred_df["label"].values

lr.fit(X, y)
calibrated_probs = lr.predict_proba(X)[:, 1]

# محاسبه مجدد متریک‌ها با احتمالات کالیبره‌شده
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score, balanced_accuracy_score
best_preds = (calibrated_probs >= 0.5).astype(int)

print(f"AUC کالیبره‌شده: {roc_auc_score(y, calibrated_probs):.3f}")
print(f"F1-Score کالیبره‌شده: {f1_score(y, best_preds, zero_division=0):.3f}")
print(f"Balanced Accuracy کالیبره‌شده: {balanced_accuracy_score(y, best_preds):.3f}")

AUC کالیبره‌شده: 0.555
F1-Score کالیبره‌شده: 0.000
Balanced Accuracy کالیبره‌شده: 0.500
